In [1]:
import pandas as pd
import ast
import re
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
import networkx as nx
from rapidfuzz import process, fuzz
from collections import Counter
import ast
import pyarrow

/Users/kadinwilkins/recipe_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


aliases

In [2]:
UNIT_ALIASES = {
    "t": "teaspoon", "t.": "teaspoon", "tsp": "teaspoon", "tsp.": "teaspoon",
    "teaspoon": "teaspoon", "teaspoons": "teaspoon",

    "T": "tablespoon", "T.": "tablespoon", "tbsp": "tablespoon", "tbsp.": "tablespoon",
    "tbs": "tablespoon", "tbs.": "tablespoon", "tablespoon": "tablespoon",
    "tablespoons": "tablespoon",

    "c": "cup", "c.": "cup", "cup": "cup", "cups": "cup",

    "oz": "ounce", "oz.": "ounce", "ounce": "ounce", "ounces": "ounce",

    "lb": "pound", "lb.": "pound", "lbs": "pound", "lbs.": "pound",
    "pound": "pound", "pounds": "pound",

    "g": "gram", "g.": "gram", "gram": "gram", "grams": "gram",

    "kg": "kilogram", "kg.": "kilogram", "kilogram": "kilogram",
    "kilograms": "kilogram",

    "ml": "milliliter", "ml.": "milliliter", "mL": "milliliter",
    "milliliter": "milliliter", "milliliters": "milliliter",

    "l": "liter", "L": "liter", "liter": "liter", "liters": "liter",

    "pinch": "pinch", "pinches": "pinch",
    "dash": "dash", "dashes": "dash",
    "clove": "clove", "cloves": "clove",
    "slice": "slice", "slices": "slice",
    "can": "can", "cans": "can",
    "package": "package", "packages": "package",
    "pkg": "package", "pkg.": "package",
}

BASE_PATH = "./data"

load state

In [3]:
recipes_df = pd.read_parquet(f"{BASE_PATH}/recipes_trimmed_keep_longest.parquet")
ingredient_matches = pd.read_parquet(f"{BASE_PATH}/accepted_ingredient_matches_v1.parquet")
ingredient_candidates = pd.read_parquet(f"{BASE_PATH}/ingredient_food_candidates_v1.parquet")

embeddings = np.load(f"{BASE_PATH}/recipe_embeddings_fp16.npy")
index = faiss.read_index(f"{BASE_PATH}/recipe_index-001.faiss")